# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [22]:
# 1. Install system dependencies
!apt-get update -qq && apt-get install -y -qq build-essential python3-dev ffmpeg libsndfile1

# 2. Clone piper-sample-generator and pin to the exact v2 commit openWakeWord expects
!rm -rf piper-sample-generator
!git clone https://github.com/rhasspy/piper-sample-generator
!git -C piper-sample-generator checkout f1988a4
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# 3. Install Python 3.12 compatible packages
!pip install -q piper-phonemize-cross webrtcvad mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0
!pip install -q speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!pip install -q pronouncing==0.2.0 deep-phonemizer==0.0.19 onnx onnxruntime soundfile librosa datasets scipy tqdm pyyaml

# 4. Clone and install openWakeWord (--no-deps avoids the broken tflite-runtime wheel on Python 3.12)
!rm -rf openwakeword
!git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword --no-deps

# 5. Download required openWakeWord backbone models
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# 6. Patch torchaudio 2.5+ removals & broken TFLite conversion in openWakeWord and torch_audiomentations
import glob

# Patch torch_audiomentations set_audio_backend call
for p in glob.glob("/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py"):
    with open(p, "r") as f:
        code = f.read()
    code = code.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
    with open(p, "w") as f:
        f.write(code)

# Patch openwakeword/data.py to use soundfile instead of removed torchaudio.info / set_audio_backend
with open("openwakeword/openwakeword/data.py", "r") as f:
    data_code = f.read()
data_code = data_code.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
# Inject a torchaudio.info shim using soundfile at the top of data.py
shim = """
import soundfile as _sf
import torchaudio as _ta
if not hasattr(_ta, 'info'):
    class _AudioMeta:
        def __init__(self, path):
            info = _sf.info(path)
            self.sample_rate = info.samplerate
            self.num_frames = info.frames
            self.num_channels = info.channels
    _ta.info = _AudioMeta
"""
data_code = shim + "\n" + data_code
with open("openwakeword/openwakeword/data.py", "w") as f:
    f.write(data_code)

# Patch openwakeword/train.py to skip broken onnx_tf TFLite conversion (retaining clean .onnx output)
with open("openwakeword/openwakeword/train.py", "r") as f:
    train_code = f.read()
train_code = train_code.replace(
    'convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),',
    '# Skipped TFLite conversion (broken on Py3.12)\n    # convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),'
)
train_code = train_code.replace(
    'os.path.join(config["output_dir"], config["model_name"] + ".tflite"))',
    '# os.path.join(config["output_dir"], config["model_name"] + ".tflite"))'
)
with open("openwakeword/openwakeword/train.py", "w") as f:
    f.write(train_code)

print("✅ Environment setup and patches applied successfully!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 184 (delta 70), reused 49 (delta 49), pack-reused 96 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 7.31 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Note: switching to 'f1988a4'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  gi

In [23]:
import os
import io
import sys
import uuid
import yaml
import glob
from pathlib import Path
import numpy as np
import torch
import scipy.io.wavfile
import soundfile as sf
import librosa
import datasets
from tqdm import tqdm

# Ensure openwakeword and piper-sample-generator are on sys.path
if os.path.abspath("./openwakeword") not in sys.path:
    sys.path.insert(0, os.path.abspath("./openwakeword"))
if os.path.abspath("./piper-sample-generator") not in sys.path:
    sys.path.insert(0, os.path.abspath("./piper-sample-generator"))

# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [24]:
# Cell 3
import os
import glob
import pandas as pd
import numpy as np
import scipy.io.wavfile
import librosa
from tqdm import tqdm

output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)

if len(glob.glob(f"{output_dir}/*.wav")) == 0:
    # Bypass datasets entirely. Download the auto-generated Parquet database directly.
    parquet_url = "https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
    !wget -q {parquet_url} -O mit_rirs.parquet

    print("Processing MIT RIRs...")
    df = pd.read_parquet("mit_rirs.parquet")

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Saving RIRs"):
        audio_dict = row['audio']
        name = audio_dict['path'].split('/')[-1]
        temp_path = os.path.join(output_dir, f"temp_{name}")

        # Write raw bytes to physical disk to bypass Python 3.13 BytesIO C-FFI bugs
        with open(temp_path, "wb") as f:
            f.write(audio_dict['bytes'])

        try:
            # Load from disk and resample
            audio_arr, _ = librosa.load(temp_path, sr=16000, mono=True)
            scipy.io.wavfile.write(
                os.path.join(output_dir, name),
                16000,
                (audio_arr * 32767).astype(np.int16)
            )
        except Exception:
            pass
        finally:
            if os.path.exists(temp_path):
                os.remove(temp_path)

print(f"✅ MIT RIR files ready: {len(glob.glob(f'{output_dir}/*.wav'))} clips")

✅ MIT RIR files ready: 1 clips


In [25]:
# Cell 4
import os
import glob
from pathlib import Path
import numpy as np
import scipy.io.wavfile
import librosa
from tqdm import tqdm

# We will use two chunks of AudioSet to get plenty of background audio
# without FMA script errors or MP3 decoding bugs.

os.makedirs("audioset", exist_ok=True)
tar_files = ["bal_train09.tar", "bal_train10.tar"]

for fname in tar_files:
    out_dir = f"audioset/{fname}"
    if not os.path.exists(out_dir):
        link = f"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}"
        !wget -q --show-progress -O {out_dir} {link}
        !cd audioset && tar -xf {fname}

output_dir_audioset = "./audioset_16k"
os.makedirs(output_dir_audioset, exist_ok=True)

flac_files = list(Path("audioset/audio").glob("**/*.flac"))
if len(glob.glob(f"{output_dir_audioset}/*.wav")) < len(flac_files):
    for flac_path in tqdm(flac_files, desc="Converting AudioSet to 16kHz"):
        name = flac_path.name.replace(".flac", ".wav")
        try:
            # librosa can natively read flac files on disk
            audio_arr, _ = librosa.load(str(flac_path), sr=16000, mono=True)
            scipy.io.wavfile.write(
                os.path.join(output_dir_audioset, name),
                16000,
                (audio_arr * 32767).astype(np.int16)
            )
        except Exception:
            pass  # Skip corrupted flacs

print(f"✅ Background audio ready: {len(glob.glob('./audioset_16k/*.wav'))} clips")

✅ Background audio ready: 0 clips


In [26]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-10-19 18:38:09--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 3.170.185.35, 3.170.185.14, 3.170.185.25, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.35|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&X-Xet-Cas-Uid=public&xip=1abhRErRInU&Expires=1792438689&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD91c2VyX2lkPXB1

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [27]:
# Cell 6
import yaml

# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Clean up any previous failed/mispronounced runs for Cassidy
!rm -rf ./my_custom_model/Cassidy

# Phonetic spelling for Piper TTS so it pronounces "Cassidy" properly
config["target_phrase"] = ["cas ih dee"]

# Keep the final output file named Cassidy.onnx
config["model_name"] = "Cassidy"

# Training parameters
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = 1500

# REPLACED: FMA removed, relying entirely on the stable AudioSet directory
config["background_paths"] = ['./audioset_16k']

config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("✅ Saved my_model.yaml for wake word:", config["model_name"])

✅ Saved my_model.yaml for wake word: Cassidy


In [28]:
# Cell 6
import yaml

# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Clean up any previous mismatched directories
!rm -rf ./my_custom_model

# Phonetic spelling for Piper TTS so it pronounces "Cassidy" properly
config["target_phrase"] = ["cas ih dee"]

# MUST exactly match the target phrase so train.py doesn't break its internal folder paths
config["model_name"] = "cas_ih_dee"

# Training parameters
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = 1500

config["background_paths"] = ['./audioset_16k']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("✅ Saved my_model.yaml for wake word:", config["model_name"])

✅ Saved my_model.yaml for wake word: cas_ih_dee


# Train the Model

In [50]:
# Cell 7: Complete Pre-Flight Patch + Training Pipeline
import os
import sys
import glob
import subprocess
import yaml
import numpy as np
import soundfile as sf
import librosa
import scipy.io.wavfile

# ---------------------------------------------------------
# 1. Align piper-sample-generator to v2.0.0 (matches en_US-libritts_r-medium.pt)
# ---------------------------------------------------------
subprocess.run(["git", "-C", "piper-sample-generator", "fetch", "--tags"], check=False)
subprocess.run(["git", "-C", "piper-sample-generator", "checkout", "v2.0.0"], check=False)

os.makedirs("piper-sample-generator/models", exist_ok=True)
med_model = os.path.abspath("piper-sample-generator/models/en_US-libritts_r-medium.pt")
high_model = os.path.abspath("piper-sample-generator/models/en-us-libritts-high.pt")

if not os.path.exists(med_model) or os.path.getsize(med_model) < 10_000_000:
    subprocess.run([
        "wget", "-q", "-O", med_model,
        "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
    ], check=True)

# Copy to high_model path in case any script references the old default
if not os.path.exists(high_model):
    subprocess.run(["cp", med_model, high_model], check=True)

# Also copy generate_samples.py into openwakeword directory so imports never fail
subprocess.run(["cp", "piper-sample-generator/generate_samples.py", "openwakeword/openwakeword/generate_samples.py"], check=True)

# ---------------------------------------------------------
# 2. Patch generate_samples.py (default model path + PyTorch 2.6+ weights_only=False)
# ---------------------------------------------------------
for gs_path in ["piper-sample-generator/generate_samples.py", "openwakeword/openwakeword/generate_samples.py"]:
    with open(gs_path, "r") as f:
        gs_code = f.read()

    # Ensure model parameter has default path
    gs_code = gs_code.replace(
        "def generate_samples(\n    text",
        f"def generate_samples(\n    text,\n    model: str = '{med_model}',"
    )
    gs_code = gs_code.replace(
        "model: Union[str, Path],",
        f"model: Union[str, Path] = '{med_model}',"
    )
    gs_code = gs_code.replace(
        "model: str,",
        f"model: str = '{med_model}',"
    )
    # Fix PyTorch 2.6+ weights_only=True crash on torch.load
    if "weights_only" not in gs_code:
        gs_code = gs_code.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)")

    with open(gs_path, "w") as f:
        f.write(gs_code)

# ---------------------------------------------------------
# 3. Patch openwakeword files for SciPy 1.15+, Torchaudio 2.5+, PyTorch 2.6+
# ---------------------------------------------------------
SITE_CUSTOMIZE_SHIM = """# Auto-injected compatibility shims for Python 3.13 / PyTorch 2.6+ / SciPy 1.15+
import torch as _torch
_orig_torch_load = _torch.load
def _safe_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(*args, **kwargs)
_torch.load = _safe_torch_load

if hasattr(_torch, 'onnx') and hasattr(_torch.onnx, 'export'):
    _orig_onnx_export = _torch.onnx.export
    def _safe_onnx_export(*args, **kwargs):
        try:
            return _orig_onnx_export(*args, dynamo=False, **kwargs)
        except TypeError:
            return _orig_onnx_export(*args, **kwargs)
    _torch.onnx.export = _safe_onnx_export

import scipy.special as _sp
if not hasattr(_sp, 'sph_harm') and hasattr(_sp, 'sph_harm_y'):
    def _sph_harm(m, n, theta, phi):
        return _sp.sph_harm_y(n, m, phi, theta)
    _sp.sph_harm = _sph_harm

import soundfile as _sf
import torchaudio as _ta
if not hasattr(_ta, 'set_audio_backend'):
    _ta.set_audio_backend = lambda *a, **kw: None
if not hasattr(_ta, 'info'):
    class _AudioMeta:
        def __init__(self, path):
            info = _sf.info(path)
            self.sample_rate = info.samplerate
            self.num_frames = info.frames
            self.num_channels = info.channels
    _ta.info = _AudioMeta
# END_SHIM
"""

for target_file in ["openwakeword/openwakeword/data.py", "openwakeword/openwakeword/train.py"]:
    with open(target_file, "r") as f:
        code = f.read()
    if "# END_SHIM" not in code:
        code = SITE_CUSTOMIZE_SHIM + "\n" + code
    with open(target_file, "w") as f:
        f.write(code)

# Fix train.py float16 validation tensor mismatch & disable broken TFLite export cleanly
with open("openwakeword/openwakeword/train.py", "r") as f:
    t_code = f.read()

# Fix validation data dtype mismatch in train.py
t_code = t_code.replace(
    "val_set_hrs = 11.3",
    "X_val_fp = X_val_fp.float() if hasattr(X_val_fp, 'float') else X_val_fp\n        val_set_hrs = 11.3"
)
t_code = t_code.replace(
    "def convert_onnx_to_tflite(onnx_model_path, output_path):",
    "def convert_onnx_to_tflite(onnx_model_path, output_path):\n    return None"
)
with open("openwakeword/openwakeword/train.py", "w") as f:
    f.write(t_code)

# ---------------------------------------------------------
# 4. Write clean my_model.yaml config
# ---------------------------------------------------------
with open("openwakeword/examples/custom_model.yml", "r") as f:
    config = yaml.load(f.read(), yaml.Loader)

subprocess.run(["rm", "-rf", "./my_custom_model"], check=False)

config["target_phrase"] = ["cas ih dee"]
config["model_name"] = "cas_ih_dee"
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = 1500
config["piper_sample_generator_path"] = os.path.abspath("./piper-sample-generator")
config["rir_paths"] = [os.path.abspath("./mit_rirs")]
config["background_paths"] = [os.path.abspath("./audioset_16k")]
config["background_paths_duplication_rate"] = [1]
config["false_positive_validation_data_path"] = os.path.abspath("validation_set_features.npy")
config["feature_data_files"] = {"ACAV100M_sample": os.path.abspath("openwakeword_features_ACAV100M_2000_hrs_16bit.npy")}

# Provide explicit custom_negative_phrases so pronouncing doesn't fail on phonetic non-dictionary words
config["custom_negative_phrases"] = [
    "capacity", "canopy", "catastrophe", "cavity", "calamity",
    "comedy", "custody", "melody", "parody", "remedy", "tragedy",
    "pass it to me", "cast a bee", "fast and free", "cas", "dee"
]

with open("my_model.yaml", "w") as f:
    yaml.dump(config, f)

# ---------------------------------------------------------
# 5. Pre-Flight Test (Verifies Piper TTS works in 5 seconds before running full job)
# ---------------------------------------------------------
print("Running 5-second pre-flight TTS check...")
sys.path.insert(0, os.path.abspath("./piper-sample-generator"))
from generate_samples import generate_samples

os.makedirs("./preflight_test", exist_ok=True)
generate_samples(
    text=["cas ih dee"],
    max_samples=1,
    batch_size=1,
    output_dir="./preflight_test",
    file_names=["test.wav"]
)
assert os.path.exists("./preflight_test/test.wav"), "Pre-flight TTS test failed to write test.wav!"
subprocess.run(["rm", "-rf", "./preflight_test"], check=False)
print("Pre-flight TTS check passed! Starting Step 1...")

# ---------------------------------------------------------
# 6. Helper to run train.py and stream live errors if anything fails
# ---------------------------------------------------------
def run_step(flag):
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{os.path.abspath('./openwakeword')}:{os.path.abspath('./piper-sample-generator')}:" + env.get("PYTHONPATH", "")
    cmd = [sys.executable, "-u", "openwakeword/openwakeword/train.py", "--training_config", "my_model.yaml", flag]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"train.py {flag} exited with code {proc.returncode}")

# Step 1: Generate Clips
run_step("--generate_clips")

wav_dirs = [
    "./my_custom_model/cas_ih_dee/positive_train",
    "./my_custom_model/cas_ih_dee/positive_test",
    "./my_custom_model/cas_ih_dee/negative_train",
    "./my_custom_model/cas_ih_dee/negative_test",
]

for d in wav_dirs:
    clips = glob.glob(f"{d}/*.wav")
    print(f"Directory {d}: {len(clips)} clips")
    assert len(clips) > 0, f"Zero clips generated in {d}!"
    for wav_path in clips:
        info = sf.info(wav_path)
        if info.samplerate != 16000:
            audio, _ = librosa.load(wav_path, sr=16000, mono=True)
            scipy.io.wavfile.write(wav_path, 16000, (audio * 32767).astype(np.int16))

# Step 2: Augment Clips
print("\nStarting Step 2: Augmenting clips...")
run_step("--augment_clips")

assert os.path.exists("./my_custom_model/cas_ih_dee/positive_features_train.npy"), "Feature extraction failed!"

# Step 3: Train Model
print("\nStarting Step 3: Training model...")
run_step("--train_model")

# Rename to Cassidy.onnx and download
original_onnx = "./my_custom_model/cas_ih_dee.onnx"
final_onnx = "./my_custom_model/Cassidy.onnx"

if os.path.exists(original_onnx):
    os.rename(original_onnx, final_onnx)
    print(f"\nSUCCESS: Saved {final_onnx} ({os.path.getsize(final_onnx)/1024:.1f} KB)")
    from google.colab import files
    files.download(final_onnx)
else:
    raise FileNotFoundError("ONNX model was not created.")

Running 5-second pre-flight TTS check...


SyntaxError: invalid syntax (generate_samples.py, line 28)

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [39]:
# Cell 7
import sys
import glob
import soundfile as sf
import librosa
import scipy.io.wavfile
import numpy as np
import IPython.display as ipd

# Step 1: Generate synthetic clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# Verify and enforce 16,000 Hz 16-bit PCM on all generated WAVs using the correct folder
wav_dirs = [
    "./my_custom_model/cas_ih_dee/positive_train",
    "./my_custom_model/cas_ih_dee/positive_test",
    "./my_custom_model/cas_ih_dee/negative_train",
    "./my_custom_model/cas_ih_dee/negative_test",
]

resampled_count = 0
for d in wav_dirs:
    for wav_path in glob.glob(f"{d}/*.wav"):
        info = sf.info(wav_path)
        if info.samplerate != 16000:
            audio, _ = librosa.load(wav_path, sr=16000, mono=True)
            scipy.io.wavfile.write(wav_path, 16000, (audio * 32767).astype(np.int16))
            resampled_count += 1

print(f"✅ Clips generated! (Resampled {resampled_count} clips to 16kHz)")

# Listen to a sample
sample_wavs = glob.glob("./my_custom_model/cas_ih_dee/positive_train/*.wav")
if sample_wavs:
    display(ipd.Audio(sample_wavs[0]))
else:
    print("⚠️ Warning: No WAV files found. Generation failed.")

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en-us-libritts-high.pt
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 676, in <module>
    generate_samples(
    ~~~~~~~~~~~~~~~~^
        text=config["target_phrase"], max_samples=config["n_samples"]-n_current_samples,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        file_names=[uuid.uuid4().hex + ".wav" for 

In [40]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 751, in <module>
    sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
                                                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "numpy/random/mtrand.pyx", line 798, in numpy.random.mtrand.RandomState.randint
  File "numpy/random/_bounded_integers.pyx", line 1334, in numpy.random._bounded_integers._rand_int64
ValueError: high <= 0


In [41]:
# Cell 9
import sys
import os

# Step 3: Train model
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Rename the resulting file to Cassidy.onnx so you get the filename you want
original_onnx = "./my_custom_model/cas_ih_dee.onnx"
final_onnx = "./my_custom_model/Cassidy.onnx"

if os.path.exists(original_onnx):
  os.rename(original_onnx, final_onnx)
  print(f"🎉 Success! Model saved and renamed to {final_onnx} ({os.path.getsize(final_onnx)/1024:.1f} KB)")
  from google.colab import files
  files.download(final_onnx)
else:
  print("❌ ONNX file was not found. Check the training output above.")

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 751, in <module>
    sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
                                                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "numpy/random/mtrand.pyx", line 798, in numpy.random.mtrand.RandomState.randint
  File "numpy/random/_bounded_integers.pyx", line 1334, in numpy.random._bounded_integers._rand_int64
ValueError: high <= 0
❌ ONNX file was not found. Check the training output above.


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!